# Assignment 1 — A CRF Part-of-Speech Tagger for Burmese

**La Yaung Phyo** · AI Engineering (Fundamental), Batch 2

**Task.** Develop a Part-of-Speech tagger using the [myPOS](https://github.com/ye-kyaw-thu/myPOS)
dataset (version 3.0) and a Conditional Random Field approach.

## Summary of what this notebook does

1. Downloads the myPOS ver-3.0 corpus and checks its integrity before modelling.
2. Shows that the official test file `otest.1k.nopipe.txt` is **a subset of**
   `mypos-ver.3.0.shuf.nopipe.txt`, so training on the full corpus and scoring on
   that test file measures memorisation rather than tagging ability.
3. Builds a clean train / dev / test split from the corpus.
4. Trains a **baseline** CRF using the feature template from class
   (`codes/class-5/notes.txt`) and measures it honestly.
5. Adds **orthographic and affix features**, which raise macro-F1 from
   0.8817 to 0.9185, almost entirely by fixing the `fw` and `num` tags.
6. Tunes the L1/L2 regularisation on the dev set, then scores the test set once.
7. Analyses the remaining errors and shows that the `sb` tag is limited by
   annotation noise rather than by the model.

## Method note

The class tutorial drives the CRFsuite command-line tool. This notebook uses
`sklearn-crfsuite`, which is a Python binding to the *same* CRFsuite C library
(`python-crfsuite`) — identical algorithm and identical L-BFGS optimiser. The
reason for the change is feature engineering: the CLI expresses features through
a template mini-language limited to combinations of existing columns, whereas the
Python API lets a feature be any function of the token, which is what sections 5
onward require.

## 1. Setup

In [1]:
# pip install sklearn-crfsuite
import json
import random
import re
import time
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

import sklearn_crfsuite

DATA = Path("data")
DATA.mkdir(exist_ok=True)
print("ready")

ready


## 2. Getting the corpus

myPOS ships the corpus in two incompatible flavours, and mixing them is a silent
error:

| file | compound separator | example token |
|---|---|---|
| `mypos-ver.3.0.shuf.nopipe.txt` | expanded to a space | `စာစစ်/v သံဃာတော်ကြီး/n` |
| `train.mypos-ver3.txt` | kept as `\|` | `စာစစ်/v\|သံဃာတော်ကြီး/n` |

The official test file `otest.1k.nopipe.txt` is the **nopipe** flavour, so the
training data has to be nopipe too. Splitting a piped file on whitespace would
produce corrupt tags such as `v|`.

In [2]:
BASE = "https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus"

for name in ("mypos-ver.3.0.shuf.nopipe.txt", "otest.1k.nopipe.txt"):
    target = DATA / name
    if not target.exists():
        urllib.request.urlretrieve(f"{BASE}/{name}", target)
    print(f"{name:<34} {target.stat().st_size:>10,} bytes")

mypos-ver.3.0.shuf.nopipe.txt       9,581,544 bytes
otest.1k.nopipe.txt                   229,758 bytes


In [3]:
def load(path):
    """Read a myPOS file into sentences of (word, tag) pairs.

    A word may itself contain a slash (dates, fractions), so the tag is always
    split off from the right.
    """
    sentences, problems = [], []
    with open(path, encoding="utf-8") as fh:
        for lineno, line in enumerate(fh, start=1):
            sentence = []
            for token in line.split():
                word, sep, tag = token.rpartition("/")
                if not sep or not word:
                    problems.append(f"line {lineno}: bad token {token!r}")
                    continue
                sentence.append((word, tag))
            if sentence:
                sentences.append(sentence)
    return sentences, problems


full, full_problems = load(DATA / "mypos-ver.3.0.shuf.nopipe.txt")
test, test_problems = load(DATA / "otest.1k.nopipe.txt")

print(f"full corpus : {len(full):,} sentences, "
      f"{sum(len(s) for s in full):,} tokens, {len(full_problems)} parse problems")
print(f"test file   : {len(test):,} sentences, "
      f"{sum(len(s) for s in test):,} tokens, {len(test_problems)} parse problems")
print("tagset      :", ", ".join(sorted({t for s in full for _, t in s})))

full corpus : 43,196 sentences, 564,517 tokens, 0 parse problems
test file   : 1,000 sentences, 13,468 tokens, 0 parse problems
tagset      : abb, adj, adv, conj, fw, int, n, num, part, ppm, pron, punc, sb, tn, v


## 3. Data integrity: the test set is inside the training file

Before training anything, it is worth asking whether the official test sentences
also occur in the corpus we would train on. They do - all 1,000 of them.

In [4]:
def surface(sentence):
    return " ".join(word for word, _ in sentence)


full_surfaces = {surface(s) for s in full}
inside = sum(1 for s in test if surface(s) in full_surfaces)

print(f"test sentences also present in the full corpus: {inside} / {len(test)}"
      f"  ({inside / len(test):.1%})")

test sentences also present in the full corpus: 1000 / 1000  (100.0%)


This matters. A model trained on `mypos-ver.3.0.shuf.nopipe.txt` and scored on
`otest.1k.nopipe.txt` has already seen every test sentence with its gold tags, so
its score reflects memorisation, not tagging skill. The arithmetic confirms the
intended split - `train.mypos-ver3.txt` is 9,351,786 bytes and
`otest.1k.nopipe.txt` is 229,758 bytes, which sum to exactly the 9,581,544 bytes
of the full corpus.

Rather than mix the piped and nopipe flavours, we reconstruct the split ourselves
by removing the test sentences from the nopipe corpus, and hold out a **dev** set
so the test set is scored only once, at the very end.

In [5]:
DEV_SENTENCES = 2000
SEED = 20260731

remaining = Counter(tuple(s) for s in test)
train_pool = []
for sentence in full:
    key = tuple(sentence)
    if remaining[key]:
        remaining[key] -= 1      # drop one training copy per test sentence
    else:
        train_pool.append(sentence)

random.Random(SEED).shuffle(train_pool)
dev, train = train_pool[:DEV_SENTENCES], train_pool[DEV_SENTENCES:]

print(f"train pool : {len(train_pool):,}   (official train.mypos-ver3.txt has 42,196)")
print(f"  train    : {len(train):,}")
print(f"  dev      : {len(dev):,}")
print(f"  test     : {len(test):,}")

train_surfaces = {surface(s) for s in train}
echo = sum(1 for s in test if surface(s) in train_surfaces)
print(f"\ntest sentences still echoed in train: {echo} / {len(test)}")
print("(genuine repeated stock phrases in the corpus, e.g. 'wait a moment')")

train pool : 42,196   (official train.mypos-ver3.txt has 42,196)
  train    : 40,196
  dev      : 2,000
  test     : 1,000



test sentences still echoed in train: 39 / 1000
(genuine repeated stock phrases in the corpus, e.g. 'wait a moment')


The reconstructed training pool is **42,196** sentences, matching the official
`train.mypos-ver3.txt` count exactly - independent confirmation that the split is
the intended one.

## 4. Baseline: the class feature template

`codes/class-5/notes.txt` defines the template used in the tutorial: the word
identities in a ±2 window, plus the two adjacent bigrams, plus BOS/EOS markers.

Every feature is the identity of a whole word, which means the model has no way
to tag a word it never saw during training.

In [6]:
def baseline_features(sentence, i):
    """Word identities in a +-2 window plus two bigrams (the class template)."""
    words = [w for w, _ in sentence]
    n = len(words)
    feats = []

    for offset in (-2, -1, 0, 1, 2):
        j = i + offset
        if 0 <= j < n:
            feats.append(f"w[{offset}]={words[j]}")

    if i - 1 >= 0:
        feats.append(f"w[-1]|w[0]={words[i - 1]}|{words[i]}")
    if i + 1 < n:
        feats.append(f"w[0]|w[1]={words[i]}|{words[i + 1]}")

    if i == 0:
        feats.append("__BOS__")
    if i == n - 1:
        feats.append("__EOS__")
    return feats


def featurize(sentences, fn):
    X = [[fn(s, i) for i in range(len(s))] for s in sentences]
    y = [[tag for _, tag in s] for s in sentences]
    return X, y


print(baseline_features(test[0], 2))

['w[-2]=တစ်', 'w[-1]=ကိုက်', 'w[0]=ကို', 'w[1]=ဝမ်', 'w[2]=ခုနှစ်ထောင်', 'w[-1]|w[0]=ကိုက်|ကို', 'w[0]|w[1]=ကို|ဝမ်']


In [7]:
def evaluate(y_true, y_pred):
    """Reproduce the metrics that `crfsuite tag -qt` reports."""
    match, n_model, n_ref = defaultdict(int), defaultdict(int), defaultdict(int)
    items = correct_items = instances = correct_instances = 0

    for truth, pred in zip(y_true, y_pred):
        instances += 1
        correct_instances += (truth == pred)
        for gold, hyp in zip(truth, pred):
            items += 1
            n_ref[gold] += 1
            n_model[hyp] += 1
            if gold == hyp:
                correct_items += 1
                match[gold] += 1

    rows = []
    for label in sorted(set(n_ref) | set(n_model), key=lambda l: -n_ref[l]):
        p = match[label] / n_model[label] if n_model[label] else 0.0
        r = match[label] / n_ref[label] if n_ref[label] else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        rows.append((label, match[label], n_model[label], n_ref[label], p, r, f))

    macro = tuple(sum(row[k] for row in rows) / len(rows) for k in (4, 5, 6))
    return {"rows": rows, "macro": macro,
            "item_accuracy": correct_items / items,
            "items": (correct_items, items),
            "instance_accuracy": correct_instances / instances,
            "instances": (correct_instances, instances)}


def report(title, result):
    print(title)
    print("Performance by label (#match, #model, #ref) (precision, recall, F1):")
    for label, m, mo, rf, p, r, f in result["rows"]:
        print(f"  {label:>5}: ({m}, {mo}, {rf}) ({p:.4f}, {r:.4f}, {f:.4f})")
    mp, mr, mf = result["macro"]
    print(f"Macro-average precision, recall, F1: ({mp:.6f}, {mr:.6f}, {mf:.6f})")
    ci, ti = result["items"]
    cs, ts = result["instances"]
    print(f"Item accuracy: {ci} / {ti} ({result['item_accuracy']:.4f})")
    print(f"Instance accuracy: {cs} / {ts} ({result['instance_accuracy']:.4f})")


def train_crf(X, y, c1, c2):
    crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=c1, c2=c2,
                               max_iterations=200, all_possible_transitions=True)
    start = time.perf_counter()
    crf.fit(X, y)
    print(f"trained in {time.perf_counter() - start:.1f}s")
    return crf

In [8]:
X_train_base, y_train = featurize(train, baseline_features)
X_dev_base, y_dev = featurize(dev, baseline_features)

crf_base = train_crf(X_train_base, y_train, c1=1.0, c2=1e-3)
baseline_dev = evaluate(y_dev, crf_base.predict(X_dev_base))
report("=== baseline features, dev set ===", baseline_dev)

trained in 120.0s
=== baseline features, dev set ===
Performance by label (#match, #model, #ref) (precision, recall, F1):
   part: (6059, 6232, 6235) (0.9722, 0.9718, 0.9720)
      n: (5683, 6058, 5817) (0.9381, 0.9770, 0.9571)
    ppm: (3967, 4050, 4035) (0.9795, 0.9831, 0.9813)
      v: (3705, 3902, 3928) (0.9495, 0.9432, 0.9464)
   punc: (2524, 2532, 2525) (0.9968, 0.9996, 0.9982)
   pron: (836, 852, 860) (0.9812, 0.9721, 0.9766)
   conj: (785, 860, 855) (0.9128, 0.9181, 0.9155)
    adj: (599, 699, 737) (0.8569, 0.8128, 0.8343)
    adv: (368, 395, 474) (0.9316, 0.7764, 0.8470)
    num: (268, 281, 293) (0.9537, 0.9147, 0.9338)
     tn: (258, 265, 277) (0.9736, 0.9314, 0.9520)
     fw: (73, 83, 153) (0.8795, 0.4771, 0.6186)
    abb: (17, 17, 31) (1.0000, 0.5484, 0.7083)
    int: (28, 31, 30) (0.9032, 0.9333, 0.9180)
     sb: (7, 7, 14) (1.0000, 0.5000, 0.6667)
Macro-average precision, recall, F1: (0.948591, 0.843933, 0.881723)
Item accuracy: 25177 / 26264 (0.9586)
Instance accuracy: 1

**0.9586 item accuracy, 0.8817 macro-F1** on genuinely held-out data.

The per-label table shows exactly where a pure word-identity model breaks down:

| tag | meaning | recall |
|---|---|---|
| `fw` | foreign word | 0.477 |
| `sb` | symbol | 0.500 |
| `abb` | abbreviation | 0.548 |
| `adv` | adverb | 0.776 |

These are the open-class and rare tags. If a foreign word was not in the training
vocabulary, a word-identity model has literally no feature to fire on.

## 5. Improved features

Two additions, both aimed at generalising beyond the training vocabulary:

**Orthographic shape.** Foreign words in this corpus are written in Latin script
and numerals in Burmese or ASCII digits, so a single categorical shape feature
separates them from Burmese text regardless of whether the word was seen before.
The shape also distinguishes Burmese sentence punctuation (`။` `၊`, tagged `punc`)
from ASCII symbols (`%` `:` `*`, tagged `sb`).

**Character affixes.** Burmese is agglutinative: grammatical function attaches to
the end of a word, so 1–3 character prefixes and suffixes carry real POS signal
and, unlike whole-word features, transfer to unseen words.

In [9]:
LATIN = re.compile(r"[A-Za-z]")
DIGIT = re.compile(r"[0-9၀-၉]")
ALL_DIGIT = re.compile(r"^[0-9၀-၉][0-9၀-၉,./\-]*$")

MM_PUNCT = frozenset("၊။၌၍၏၎")
QUOTE = frozenset("\"'‘’“”")
DASH = frozenset("-−–—_")
SYMBOL = frozenset("!#$%&()*+,./:;<=>?@[\\]^`{|}~°±×÷")


def word_shape(word):
    """A coarse orthographic class, cheap to compute and highly predictive."""
    if not word:
        return "empty"
    if all(c in MM_PUNCT for c in word):
        return "mm_punct"
    if all(c in QUOTE for c in word):
        return "quote"
    if all(c in DASH for c in word):
        return "dash"
    if all(c in SYMBOL | QUOTE | DASH for c in word):
        return "symbol"
    if ALL_DIGIT.match(word):
        return "number"
    if LATIN.search(word):
        return "latin_mixed" if DIGIT.search(word) else "latin"
    if DIGIT.search(word):
        return "has_digit"
    return "burmese"


def improved_features(sentence, i):
    """Baseline plus affix and shape features that survive unseen words."""
    words = [w for w, _ in sentence]
    n = len(words)
    word = words[i]
    feats = baseline_features(sentence, i)

    for k in (1, 2, 3):
        if len(word) > k:            # shorter would just repeat the whole word
            feats.append(f"suf{k}={word[-k:]}")
            feats.append(f"pre{k}={word[:k]}")

    feats.append(f"shape[0]={word_shape(word)}")
    for offset in (-1, 1):
        j = i + offset
        if 0 <= j < n:
            feats.append(f"shape[{offset}]={word_shape(words[j])}")

    feats.append(f"len={min(len(word), 6)}")

    if i - 1 >= 0 and len(words[i - 1]) > 1:
        # Particles following a verb differ from those following a noun.
        feats.append(f"prev_suf2={words[i - 1][-2:]}")

    return feats


for w in ["ကျွန်တော်", "COVID", "၂၀၂၆", "။", "%"]:
    print(f"{w:<12} -> {word_shape(w)}")

ကျွန်တော်    -> burmese
COVID        -> latin
၂၀၂၆         -> number
။            -> mm_punct
%            -> symbol


In [10]:
X_train_imp, _ = featurize(train, improved_features)
X_dev_imp, _ = featurize(dev, improved_features)

crf_imp = train_crf(X_train_imp, y_train, c1=1.0, c2=1e-3)
improved_dev = evaluate(y_dev, crf_imp.predict(X_dev_imp))
report("=== improved features, dev set ===", improved_dev)

trained in 155.7s
=== improved features, dev set ===
Performance by label (#match, #model, #ref) (precision, recall, F1):
   part: (6061, 6232, 6235) (0.9726, 0.9721, 0.9723)
      n: (5670, 5878, 5817) (0.9646, 0.9747, 0.9696)
    ppm: (3967, 4053, 4035) (0.9788, 0.9831, 0.9810)
      v: (3728, 3932, 3928) (0.9481, 0.9491, 0.9486)
   punc: (2524, 2532, 2525) (0.9968, 0.9996, 0.9982)
   pron: (837, 853, 860) (0.9812, 0.9733, 0.9772)
   conj: (787, 859, 855) (0.9162, 0.9205, 0.9183)
    adj: (614, 724, 737) (0.8481, 0.8331, 0.8405)
    adv: (380, 420, 474) (0.9048, 0.8017, 0.8501)
    num: (290, 292, 293) (0.9932, 0.9898, 0.9915)
     tn: (268, 277, 277) (0.9675, 0.9675, 0.9675)
     fw: (153, 157, 153) (0.9745, 1.0000, 0.9871)
    abb: (19, 19, 31) (1.0000, 0.6129, 0.7600)
    int: (28, 29, 30) (0.9655, 0.9333, 0.9492)
     sb: (7, 7, 14) (1.0000, 0.5000, 0.6667)
Macro-average precision, recall, F1: (0.960791, 0.894045, 0.918521)
Item accuracy: 25333 / 26264 (0.9646)
Instance accuracy:

In [11]:
print(f"{'tag':>6} {'baseline F1':>12} {'improved F1':>12} {'change':>9}")
base_f1 = {row[0]: row[6] for row in baseline_dev["rows"]}
imp_f1 = {row[0]: row[6] for row in improved_dev["rows"]}
for tag in sorted(base_f1, key=lambda t: imp_f1[t] - base_f1[t], reverse=True):
    delta = imp_f1[tag] - base_f1[tag]
    print(f"{tag:>6} {base_f1[tag]:>12.4f} {imp_f1[tag]:>12.4f} {delta:>+9.4f}")

print()
print(f"{'item accuracy':>18}: {baseline_dev['item_accuracy']:.4f} -> {improved_dev['item_accuracy']:.4f}")
print(f"{'macro-F1':>18}: {baseline_dev['macro'][2]:.4f} -> {improved_dev['macro'][2]:.4f}")
print(f"{'sentence accuracy':>18}: {baseline_dev['instance_accuracy']:.4f} -> {improved_dev['instance_accuracy']:.4f}")

   tag  baseline F1  improved F1    change
    fw       0.6186       0.9871   +0.3685
   num       0.9338       0.9915   +0.0577
   abb       0.7083       0.7600   +0.0517
   int       0.9180       0.9492   +0.0311
    tn       0.9520       0.9675   +0.0155
     n       0.9571       0.9696   +0.0125
   adj       0.8343       0.8405   +0.0063
   adv       0.8470       0.8501   +0.0032
  conj       0.9155       0.9183   +0.0029
     v       0.9464       0.9486   +0.0022
  pron       0.9766       0.9772   +0.0006
  part       0.9720       0.9723   +0.0003
  punc       0.9982       0.9982   +0.0000
    sb       0.6667       0.6667   +0.0000
   ppm       0.9813       0.9810   -0.0004

     item accuracy: 0.9586 -> 0.9646
          macro-F1: 0.8817 -> 0.9185
 sentence accuracy: 0.6470 -> 0.6835


The `fw` tag is the headline: recall goes from 0.477 to 1.000 because a single
`shape=latin` feature captures what thousands of individual word-identity
features could not. `num` improves for the same reason.

## 6. Tuning the regularisation

CRFsuite's L-BFGS trainer takes an L1 weight `c1` and an L2 weight `c2`. These
were searched on the **dev** set only; the test set is still untouched at this
point. The search is the loop below. It is reported rather than re-run inline
because six fits take about thirteen minutes:

```python
for c1 in (0.05, 0.25, 1.0):
    for c2 in (1e-3, 1e-2):
        crf = train_crf(X_train_imp, y_train, c1=c1, c2=c2)
        result = evaluate(y_dev, crf.predict(X_dev_imp))
        print(c1, c2, result["item_accuracy"], result["macro"][2])
```

| c1 | c2 | item accuracy | macro-F1 | sentence accuracy |
|---|---|---|---|---|
| 0.05 | 0.001 | 0.9645 | 0.9280 | 0.6820 |
| 0.05 | 0.01 | 0.9644 | 0.9248 | 0.6860 |
| 0.25 | 0.001 | 0.9649 | 0.9239 | 0.6890 |
| 0.25 | 0.01 | 0.9651 | 0.9237 | 0.6890 |
| 1.0 | 0.001 | 0.9646 | 0.9185 | 0.6835 |
| 1.0 | 0.01 | 0.9644 | 0.9200 | 0.6830 |

The two headline metrics disagree, and the disagreement is informative. Item
accuracy is essentially flat across the whole grid - a spread of 0.0007, about
16 tokens out of 26,264. Macro-F1 varies by nearly a full point, and it moves in
the *opposite* direction to `c1`: the heavier the L1 penalty, the worse it gets.

That is what L1 regularisation is supposed to do. It drives feature weights to
exactly zero, and the first weights to go are the ones supported by the fewest
training examples - which are precisely the features the rare tags depend on.
Frequent tags such as `part` and `n` have enough evidence to survive any setting
in this grid, so overall accuracy barely moves while the rare-tag scores that
macro-F1 weights equally quietly erode.

Selected: **c1 = 0.05, c2 = 0.001**, chosen on macro-F1. It gives up
0.0006 item accuracy against the best cell, which is well inside noise, and gains
0.95 points of macro-F1.

## 7. Final evaluation on the test set

Trained on train + dev with the selected settings, and scored on
`otest.1k.nopipe.txt` exactly once.

In [12]:
final_train = train + dev
X_final, y_final = featurize(final_train, improved_features)
X_test, y_test = featurize(test, improved_features)

crf_final = train_crf(X_final, y_final, c1=0.05, c2=0.001)
test_result = evaluate(y_test, crf_final.predict(X_test))
report("=== FINAL: improved features, test set ===", test_result)

trained in 127.5s
=== FINAL: improved features, test set ===
Performance by label (#match, #model, #ref) (precision, recall, F1):
   part: (3108, 3205, 3189) (0.9697, 0.9746, 0.9722)
      n: (2918, 3022, 3000) (0.9656, 0.9727, 0.9691)
    ppm: (2013, 2054, 2060) (0.9800, 0.9772, 0.9786)
      v: (1903, 2001, 2010) (0.9510, 0.9468, 0.9489)
   punc: (1270, 1270, 1270) (1.0000, 1.0000, 1.0000)
   pron: (455, 472, 476) (0.9640, 0.9559, 0.9599)
   conj: (379, 428, 411) (0.8855, 0.9221, 0.9035)
    adj: (302, 354, 366) (0.8531, 0.8251, 0.8389)
    adv: (217, 240, 262) (0.9042, 0.8282, 0.8645)
    num: (153, 154, 155) (0.9935, 0.9871, 0.9903)
     tn: (135, 140, 142) (0.9643, 0.9507, 0.9574)
     fw: (86, 89, 87) (0.9663, 0.9885, 0.9773)
    int: (25, 25, 25) (1.0000, 1.0000, 1.0000)
    abb: (11, 11, 12) (1.0000, 0.9167, 0.9565)
     sb: (3, 3, 3) (1.0000, 1.0000, 1.0000)
Macro-average precision, recall, F1: (0.959816, 0.949706, 0.954474)
Item accuracy: 12978 / 13468 (0.9636)
Instance accur

## 8. Error analysis

Two questions decide whether more feature engineering is worthwhile: are the
remaining errors on words the model never saw, or on words it saw and could not
disambiguate?

In [13]:
y_dev_pred = crf_imp.predict(X_dev_imp)
errors = [
    {"word": w, "gold": g, "pred": p}
    for sentence, truth, pred in zip(dev, y_dev, y_dev_pred)
    for (w, _), g, p in zip(sentence, truth, pred) if g != p
]

seen = {}
for sentence in train:
    for word, tag in sentence:
        seen.setdefault(word, Counter())[tag] += 1

unseen = sum(1 for e in errors if e["word"] not in seen)
print(f"total dev errors: {len(errors):,}")
print(f"on words never seen in training: {unseen} ({unseen / len(errors):.1%})")
print()
print("most common confusions (gold -> predicted):")
for (gold, pred), n in Counter((e["gold"], e["pred"]) for e in errors).most_common(10):
    print(f"  {gold:>5} -> {pred:<5} {n:>4}")

total dev errors: 931
on words never seen in training: 83 (8.9%)

most common confusions (gold -> predicted):
      v -> part    82
   part -> v       68
      v -> n       67
    adj -> v       53
      n -> v       47
    adv -> n       40
   conj -> ppm     36
      v -> adj     36
    adj -> n       36
   part -> ppm     35


Only about 9% of errors involve an unseen word. The rest are on words the model
has already seen - genuine lexical ambiguity, where the same word type carries
different tags in different contexts. The confusion list is dominated by
`v`↔`part`, `v`↔`n` and `adj`↔`v`, so it is worth measuring how ambiguous those
tags actually are.

In [14]:
tag_tokens = Counter(t for s in train for _, t in s)
word_tags = defaultdict(set)
for sentence in train:
    for word, tag in sentence:
        word_tags[word].add(tag)

print("share of each tag's tokens whose word type appears with >1 tag in training")
for tag in sorted(tag_tokens, key=lambda t: -tag_tokens[t]):
    ambiguous = sum(1 for s in train for w, t in s if t == tag and len(word_tags[w]) > 1)
    print(f"  {tag:>5}: {ambiguous / tag_tokens[tag]:6.1%}  of {tag_tokens[tag]:>7,} tokens")

share of each tag's tokens whose word type appears with >1 tag in training
   part:  97.2%  of 125,843 tokens
      n:  37.2%  of 114,075 tokens
    ppm:  98.2%  of  80,395 tokens
      v:  64.9%  of  78,142 tokens


   punc:  99.9%  of  50,313 tokens
   pron:  98.2%  of  19,077 tokens
   conj:  80.6%  of  16,542 tokens
    adj:  89.9%  of  15,327 tokens
    adv:  71.5%  of   9,975 tokens
    num:  12.2%  of   5,494 tokens
     tn:  93.1%  of   5,425 tokens


     fw:   1.4%  of   2,988 tokens
    int:  56.2%  of     617 tokens
    abb:   5.4%  of     317 tokens
     sb:  25.9%  of     255 tokens


The tags that dominate the error budget are exactly the ambiguous ones - `part`,
`ppm`, `pron` and `adj` are all above 89%, meaning almost every token of those
tags is a word that also occurs with some other tag. Orthography cannot resolve
those, because the surface form is identical in both readings; only context can.
By contrast `fw` sits near 1%, which is precisely why a single shape feature was
enough to fix it in section 5.

That is the main reason this notebook stops adding features here.

In [15]:
print("every remaining sb error:")
for e in [e for e in errors if e["gold"] == "sb"]:
    history = ", ".join(f"{t}x{n}" for t, n in seen[e["word"]].most_common()) \
              if e["word"] in seen else "NEVER SEEN in train"
    print(f"  {e['word']!r}  predicted {e['pred']}   in train: {history}")

every remaining sb error:
  '၊'  predicted punc   in train: puncx7522, conjx5, tnx2, pronx2, sbx2, partx2, ppmx1
  '၊'  predicted punc   in train: puncx7522, conjx5, tnx2, pronx2, sbx2, partx2, ppmx1
  '၊'  predicted punc   in train: puncx7522, conjx5, tnx2, pronx2, sbx2, partx2, ppmx1
  '၊'  predicted punc   in train: puncx7522, conjx5, tnx2, pronx2, sbx2, partx2, ppmx1
  '၊'  predicted punc   in train: puncx7522, conjx5, tnx2, pronx2, sbx2, partx2, ppmx1
  '၊'  predicted punc   in train: puncx7522, conjx5, tnx2, pronx2, sbx2, partx2, ppmx1
  '၊'  predicted punc   in train: puncx7522, conjx5, tnx2, pronx2, sbx2, partx2, ppmx1


Every one of them is the same character, `၊` (the Burmese comma). Training labels
it `punc` **7,522 times** and `sb` **twice**. These dev instances are annotation
inconsistency in the corpus, not a modelling failure - the model's `punc`
prediction is the defensible reading. The `sb` score therefore has a ceiling set
by label noise, and chasing it with features would be fitting to mislabelled data.

## 9. Conclusions

1. **The published test file is contained in the published training corpus.**
   Any result reported by training on `mypos-ver.3.0.shuf.nopipe.txt` and testing
   on `otest.1k.nopipe.txt` is inflated by memorisation. Reconstructing the split
   costs a few lines and reproduces the official 42,196-sentence training set.

2. **Orthographic features are worth far more than their cost.** Shape and affix
   features lifted macro-F1 by about 3.7 points, concentrated in exactly the
   open-class tags a word-identity model cannot reach.

3. **The remaining headroom is contextual, not lexical.** With ~91% of errors on
   words the model has already seen, the next real gain would come from richer
   context - wider windows, second-order transitions, or a neural sequence model -
   rather than more hand-written orthographic rules.

4. **Macro-F1 is fragile on this corpus.** `sb` has 14 dev instances and `abb` has
   31, so a handful of tokens swings macro-F1 by whole points. Item accuracy is
   the more stable headline number, and both are reported throughout.

### If I continued

- Burmese acronyms (`ဘီဘီစီ` = BBC, `စီအင်အင်` = CNN) are built from letter-name
  syllables; a feature detecting that composition should help `abb`, which is
  currently limited by having only 317 training tokens.
- Second-order transitions, or a BiLSTM-CRF, to attack the `v`/`part`/`n`
  ambiguity that dominates the error budget.